## headers of duplicated sequences are concatenated using "#" as separator
- check how this process was done and investigate where problems exist
- only test code in this file

In [ ]:
from Bio import SeqIO
from collections import defaultdict
import gzip

def merge_fasta_headers(input_fasta, output_fasta, separator="#"):
    # Dictionary to map sequences to headers
    sequence_to_headers = defaultdict(list)

    if input_fasta[-3:] == '.gz':
        # Read the input fasta file
        with gzip.open(input_fasta, "rt") as handle:
            for record in SeqIO.parse(handle, "fasta"):
                sequence_to_headers[str(record.seq)].append(record.id)
    else:
        for record in SeqIO.parse(input_fasta, "fasta"):
            sequence_to_headers[str(record.seq)].append(record.id)

    # Write the modified fasta to the output file
    with open(output_fasta, "w") as out_fasta:
        for seq, headers in sequence_to_headers.items():
            # Combine headers with '#'
            merged_header = f"{separator}".join(headers)
            # Write to the output fasta
            out_fasta.write(f">{merged_header}\n{seq}\n")

# Example usage
input_fasta = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/final_design/results/final_design/design.fa.gz"
output_fasta = "output.fasta"
merge_fasta_headers(input_fasta, output_fasta, separator='#')

### Split the headers by separator

In [ ]:
import pandas as pd
from Bio import SeqIO
import gzip

input_fasta_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/no_duplicated_sequences_design.fa"

def read_zipped_fasta(fasta_file):
    """
    Read zipped fasta file with Biopython.
    """
    handle = gzip.open(fasta_file, 'rt')
    fasta_sequences = SeqIO.parse(handle,'fasta')
    return fasta_sequences


def fasta_to_dataframe(fasta_file, columns=[]):
    """
    Convert a fasta file to a pandas dataframe.
    """
    # case for ziped files:
    if fasta_file.endswith('.gz'):
        fasta_sequences = read_zipped_fasta(fasta_file)
    else:
        fasta_sequences = SeqIO.parse(open(fasta_file),'fasta')
    header = []
    sequence = []
    for fasta in fasta_sequences:
        header.append(fasta.id)
        sequence.append(str(fasta.seq))
    df = pd.DataFrame({'header': header, 'sequence': sequence})
    if columns != [] and len(columns) == 2:
        df.columns = columns
    return df


# Function to split the IDs and create new rows while conserving all columns
def split_ids(row, id_col, separator=';'):
    ids = row[id_col].split(separator)
    new_rows = []
    for id in ids:
        new_row = row.copy()
        new_row[id_col] = id
        new_rows.append(new_row)
    return pd.DataFrame(new_rows)

input_fasta_df = fasta_to_dataframe(input_fasta_path)
print('Number of sequences before splitting: ', input_fasta_df.shape[0])
# Apply the function to each row and concatenate the results
input_fasta_df_split = pd.concat(input_fasta_df.apply(lambda row: split_ids(row, id_col='header', separator="#"), axis=1).values)

# Reset the index
input_fasta_df_split.reset_index(drop=True, inplace=True)

print('Number of sequences after splitting: ', input_fasta_df_split.shape[0])


Number of sequences before splitting:  80215
Number of sequences after splitting:  80806


,header,sequence
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...


### Are all the headers unique?

In [9]:
input_fasta_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/no_duplicated_sequences_design.fa"

input_fasta_df = fasta_to_dataframe(input_fasta_path)
print('Number of sequences before splitting: ', input_fasta_df.shape[0])

input_fasta_df['header'].nunique()

Number of sequences before splitting:  80215


79496

In [10]:
input_fasta_df.loc[input_fasta_df.duplicated(subset='header')]

,header,sequence
76620,C_positive_heart_AB:SKI-ENST00000378536.5,AGGACCGGATCAACTAGGTTCCAGCCTCCACTGGGGAAGTGAGGGG...
76621,C_positive_heart_AB:SKI-ENST00000378536.5,AGGACCGGATCAACTCACACCATTGCGTCCCTGCGCCCGCAGGCCT...
76622,C_positive_heart_AB:SKI-ENST00000378536.5,AGGACCGGATCAACTTCAGCGCTCCACGGCCCCGGGGCGGAGGTCA...
76623,C_positive_heart_AB:SKI-ENST00000378536.5,AGGACCGGATCAACTCCGCCAGGGGCCGGCGGGCGGGGCGGGGCCG...
76624,C_positive_heart_AB:SKI-ENST00000378536.5,AGGACCGGATCAACTACCCCAGGTCACCGTGTGGCGTCCCGGTAGT...
...,...,...
77522,C_positive_heart_AB:FLNA-ENST00000420627.5,AGGACCGGATCAACTTCAGACCCCAGGCCAGACCCTCCCACTGGGA...
77523,C_positive_heart_AB:FLNA-ENST00000420627.5,AGGACCGGATCAACTGGTCGCCCATTCCCAAGCTCCCACCTTGACG...
77524,C_positive_heart_AB:FLNA-ENST00000420627.5,AGGACCGGATCAACTGGGCCCAACCAAGGAACCTGGCCTGGTCTCA...
77525,C_positive_heart_AB:FLNA-ENST00000420627.5,AGGACCGGATCAACTCCACTCGCCCTGAGTCCACACAAGTTCCTGG...


In [ ]:
cat /fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates.fa | \
sed 's/ /:/g' | \
sed '0,/>GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3/s//>GC_Mohlke$
sed '0,/>GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3/s//>GC_Mohlke$
sed '0,/>GC_Mendelian_variants:REF_chr8:11703890AG\*A|GATA4/s//>GC_Mendelian_variants:REF_chr8:11703890AG\*A|GATA4_headerDuplicate1_2/' | \
sed '0,/>GC_Mendelian_variants:REF_chr8:11703890AG\*A|GATA4/s//>GC_Mendelian_variants:REF_chr8:11703890AG\*A|GATA4_headerDuplicate2_2/' | \
sed '0,/>GC_Mendelian_variants:REF_chr8:11703860G\*T|GATA4/s//>GC_Mendelian_variants:REF_chr8:11703860G\*T|GATA4_headerDuplicate1_2/' | \
sed '0,/>GC_Mendelian_variants:REF_chr8:11703860G\*T|GATA4/s//>GC_Mendelian_variants:REF_chr8:11703860G\*T|GATA4_headerDuplicate2_2/' \
> /fast/groups/ag_kircher/MPRA/IGVF_Y1_design/resources/association_data/design_no_duplicates_sequence_and_header.fa

In [ ]:
# AGGACCGGATCAACTCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCTCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGCATTGCGTGAACCGA
# AGGACCGGATCAACTCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGCATTGCGTGAACCGA

In [ ]:
zcat design.fa.gz | grep -B 1 "AGGACCGGATCAACTCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGCATTGCGTGAACCGA"

>GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4
AGGACCGGATCAACTCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGCATTGCGTGAACCGA
--
>GC_Mendelian_variants:ALT_chr8:11703890AG>A|GATA4_chr8:11703890AG>A|GATA4
AGGACCGGATCAACTCCAGGAACTAGCATCCAGCCGGGCACCCCGGGTGACCCAGTGCCCCACACAAGATCGAGAGTTGAGCCCAAGAGGTCACCTTCTTCTCTACTGGCCCCGCCCCTCGCCCGCCGCTGCGGGATGAGGACCACAGGAAGGGGGGGCGGGGAGGGAGAAAGGGAACTCATTAATAAAGCTGACCCTGGGCACCACAGCGAACCCAATCGACCTCCGGCTGGGTTGCGGGTGATTCCCCGCTCCCTGGCGGTAGCACTTGGGCATTTTCCGCGGCATTGCGTGAACCGA